# 📊 Data Wrangling Library - Comprehensive Guide


We'll be utilizing the famous Titanic dataset to demonstrate data sanitization, structural analysis, intelligent relationship mapping, and advanced multi-variate statistical testing.

## 1. ⚙️ Initialization & Setup
First, we import our main components (`DataWrangler` and `PlottingMethods`) from the library. The library architecture utilizes Mixins to cleanly separate state management from inspection logic.

In [6]:
import pandas as pd
from data_wrangling.core import DataInspector
from data_wrangling import PlottingMethods

# Initialize the engine
dataset = DataInspector()

## 2. 📥 Data Ingestion & State Management
The library securely handles loading local CSV/JSON files, or directly passing a Pandas DataFrame. We will load the local `titanic.csv` dataset and preview it.

In [29]:
# Download Dataset File
import sys
import os
import urllib.request

# Your GitHub URL and target filename
github_url = "https://raw.githubusercontent.com/janithcyapa/Statistical-Learning-e20452/refs/heads/main/titanic.csv"
file_name = "titanic.csv"

if 'google.colab' in sys.modules:
    urllib.request.urlretrieve(github_url, file_name)
    file_path = f"/content/{file_name}"
else:
    file_path = file_name
    if not os.path.exists(file_path):
        print(f"⚠️ Warning: '{file_path}' not found in the current local directory.")

In [30]:
# Load data
dataset.load_data(file_path)

# Verify data loaded safely
dataset.df.head(3)


✅ File 'titanic.csv' loaded successfully!
--- Data Summary  ---


,Column Name,Data Type,Total Records,Missing Values
0,PassengerId,int64,891,0
1,Survived,int64,891,0
2,Pclass,int64,891,0
3,Name,object,891,0
4,Sex,object,891,0
5,Age,float64,891,177
6,SibSp,int64,891,0
7,Parch,int64,891,0
8,Ticket,float64,891,230
9,Fare,float64,891,0


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,count
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,NaN,7.2500,NaN,S,1
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,NaN,71.2833,C85,C,1
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,NaN,7.9250,NaN,S,1


## 3. 🔍 Structural Analysis & Summary
To begin our EDA, we generate a high-level summary of all column dtypes, missing values, and detailed statistical distributions (Means, Maxes, Frequent occurrences).

In [8]:
# Using detailed=True computes full numerical and categorical metrics
summary_df = dataset.get_summary(detailed=True)

--- Data Summary (Detailed) ---


,Column Name,Data Type,Total Records,Missing Values,Min,Max,Mean,Std Dev,Unique Values,Most Frequent
0,PassengerId,int64,891,0,1.00,8.910000e+02,446.000000,257.353842,NaN,NaN
1,Survived,int64,891,0,0.00,1.000000e+00,0.383838,0.486592,NaN,NaN
2,Pclass,int64,891,0,1.00,3.000000e+00,2.308642,0.836071,NaN,NaN
3,Name,object,891,0,NaN,NaN,NaN,NaN,891.0,"Abbing, Mr. Anthony"
4,Sex,object,891,0,NaN,NaN,NaN,NaN,2.0,male
5,Age,float64,891,177,0.42,8.000000e+01,29.699118,14.526497,NaN,NaN
6,SibSp,int64,891,0,0.00,8.000000e+00,0.523008,1.102743,NaN,NaN
7,Parch,int64,891,0,0.00,6.000000e+00,0.381594,0.806057,NaN,NaN
8,Ticket,float64,891,230,693.00,3.101298e+06,260318.549168,471609.268688,NaN,NaN
9,Fare,float64,891,0,0.00,5.123292e+02,32.204208,49.693429,NaN,NaN


## 4. 🧹 Missing Data Diagnostics & Imputation
We scan for structural damage and cleanly impute missing records using statistically sound strategies.

In [9]:
# 1. Locate all missing records across the rows
dataset.get_missing_data(axis='rows').head(3)

# 2. Safely impute missing 'Age' using the 'median' strategy
dataset.modify_missing_values(columns=['Age'], strategy='median')

# 3. Impute categorical 'Embarked' using the 'mode' strategy
dataset.modify_missing_values(columns=['Embarked'], strategy='mode')

🔍 Found 760 rows with missing values.


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,count
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,NaN,7.2500,NaN,S,1
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,NaN,71.2833,C85,C,1
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,NaN,7.9250,NaN,S,1
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450.0,8.0500,NaN,S,1
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877.0,8.4583,NaN,Q,1


🛠️ Missing values imputed using 'median' strategy for: ['Age']
🛠️ Missing values imputed using 'mode' strategy for: ['Embarked']


## 5. 🗑️ Cleaning Duplicates & Outliers
We provide automated tools to scan for identical records, and an IQR bounding-box algorithm for outlier tracking.

In [10]:
# Clear exact duplicates
dataset.modify_duplicates()

# Scan for outliers across numerical features. If find_and_delete=True, they will be removed.
dataset.modify_outliers(find_and_delete=True)

✨ Removed 0 duplicate rows. New row count: 891
🚨 Age: Found 66 outliers.
🚨 SibSp: Found 46 outliers.
🚨 Parch: Found 213 outliers.
🚨 Ticket: Found 16 outliers.
🚨 Fare: Found 116 outliers.
🗑️ Deleted 321 outlier rows total.


## 6. ⚖️ Feature Engineering & Normalization
Machine learning models require balanced, scaled inputs. We apply multiple transformation techniques natively.

In [20]:
# Normalize the 'Fare' column using Standard Z-Score scaling. 
# replace=False appends the new column rather than destroying the old one.
dataset.modify_normalize_data(columns=['Fare'], method='standard', replace=False)

dataset.modify_normalize_data(columns=['Age'], method='robust', replace=False)

# One-Hot Encode 'Sex'
dataset.modify_normalize_data(columns=['Sex'], method='onehot', replace=False)

dataset.df.head(3)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,count,Fare_standard,Sex_female,Sex_male,Age_robust
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,NaN,7.250,NaN,S,1,-0.613051,0.0,1.0,-0.750
1,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,NaN,7.925,NaN,S,1,-0.559685,1.0,0.0,-0.250
2,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803.0,53.100,C123,S,1,3.011874,1.0,0.0,0.875


## 7. 📊 Comprehensive Univariate Dashboard
To visualize every column's distribution instantly, the library generates an intelligent subplot dashboard. Categoricals are mapped to Histograms/Pies, Numericals to Violins/Distplots.

In [21]:
# Generate a combined summary plot for the entire dataset
# Warning: For larger datasets, you may want to limit this using columns=[]
dataset.summary_plot(
    separate_plots=False,
    columns=['Age_robust','Pclass','Fare_standard','Sex'], 
    numeric_plots=['violin', 'scatter', 'distplot'],
    categorical_plots=['pie', 'histogram'],
    subplot_cols=1)

## 8. 🧠 Intelligent Relationship Mapping
The library automatically routes to the best chart based on the columns' dtypes.

In [22]:
# 1. Numeric vs. Numeric -> Scatter with Trendline
dataset.plot_relationship('Age', 'Fare')

# 2. Categorical vs. Numeric -> Box Plot comparing medians
dataset.plot_relationship('Survived', 'Age')

# 3. Categorical vs. Categorical -> Grouped Bar Chart
dataset.plot_relationship('Survived', 'Pclass')

## 9. 📈 Advanced Correlation & Associations
Instead of just relying on Pearson, the tool implements multi-method association metrics to handle mixed dtypes.

In [24]:
# Num vs Num -> Pearson's r
dataset.plot_correlation('Age', 'Fare')

# Cat vs Num -> One-Way ANOVA Eta / Point-Biserial
dataset.plot_correlation('Survived', 'Age')

# Cat vs Cat -> Cramér's V (Contingency Chi-Square)
dataset.plot_correlation('Sex', 'Survived')

/home/jazz/.local/lib/python3.14/site-packages/plotly/express/_core.py:1985: FutureWarning:

When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.



'Point-Biserial (r = -0.523, p = 0.0000)'

### The Global Association Heatmap
By unifying all these metrics together, we can generate a single universal heatmap that shows the association strength between every variable.

In [25]:
# Automatically calculates correlations for all column combinations
dataset.plot_all_associations_heatmap()

--- Global Association Matrix ---


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,count,Fare_standard,Sex_female,Sex_male,Age_robust
PassengerId,1.000,0.058,0.052,1.0,0.112,0.054,0.090,0.0,0.076,0.018,0.932,0.049,0.0,0.018,0.112,0.112,0.054
Survived,0.058,1.000,0.241,1.0,0.523,0.015,0.116,0.0,0.175,0.269,0.955,0.150,0.0,0.269,0.523,0.523,0.015
Pclass,0.052,0.241,1.000,1.0,0.036,0.367,0.095,0.0,0.418,0.670,1.000,0.237,0.0,0.670,0.036,0.036,0.367
Name,1.000,1.000,1.000,1.0,1.000,1.000,1.000,0.0,1.000,1.000,1.000,1.000,0.0,1.000,1.000,1.000,1.000
Sex,0.112,0.523,0.036,1.0,1.000,0.073,0.205,0.0,0.037,0.070,0.897,0.213,0.0,0.070,1.000,1.000,0.073
Age,0.054,0.015,0.367,1.0,0.073,1.000,0.029,0.0,0.125,0.266,0.944,0.111,0.0,0.266,0.073,0.073,1.000
SibSp,0.090,0.116,0.095,1.0,0.205,0.029,1.000,0.0,0.089,0.368,1.000,0.055,0.0,0.368,0.205,0.205,0.029
Parch,0.000,0.000,0.000,0.0,0.000,0.000,0.000,1.0,0.000,0.000,0.000,0.000,0.0,0.000,0.000,0.000,0.000
Ticket,0.076,0.175,0.418,1.0,0.037,0.125,0.089,0.0,1.000,0.394,0.971,0.435,0.0,0.394,0.037,0.037,0.125
Fare,0.018,0.269,0.670,1.0,0.070,0.266,0.368,0.0,0.394,1.000,0.999,0.166,0.0,1.000,0.070,0.070,0.266


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,count,Fare_standard,Sex_female,Sex_male,Age_robust
PassengerId,1.000,0.058,0.052,1.0,0.112,0.054,0.090,0.0,0.076,0.018,0.932,0.049,0.0,0.018,0.112,0.112,0.054
Survived,0.058,1.000,0.241,1.0,0.523,0.015,0.116,0.0,0.175,0.269,0.955,0.150,0.0,0.269,0.523,0.523,0.015
Pclass,0.052,0.241,1.000,1.0,0.036,0.367,0.095,0.0,0.418,0.670,1.000,0.237,0.0,0.670,0.036,0.036,0.367
Name,1.000,1.000,1.000,1.0,1.000,1.000,1.000,0.0,1.000,1.000,1.000,1.000,0.0,1.000,1.000,1.000,1.000
Sex,0.112,0.523,0.036,1.0,1.000,0.073,0.205,0.0,0.037,0.070,0.897,0.213,0.0,0.070,1.000,1.000,0.073
Age,0.054,0.015,0.367,1.0,0.073,1.000,0.029,0.0,0.125,0.266,0.944,0.111,0.0,0.266,0.073,0.073,1.000
SibSp,0.090,0.116,0.095,1.0,0.205,0.029,1.000,0.0,0.089,0.368,1.000,0.055,0.0,0.368,0.205,0.205,0.029
Parch,0.000,0.000,0.000,0.0,0.000,0.000,0.000,1.0,0.000,0.000,0.000,0.000,0.0,0.000,0.000,0.000,0.000
Ticket,0.076,0.175,0.418,1.0,0.037,0.125,0.089,0.0,1.000,0.394,0.971,0.435,0.0,0.394,0.037,0.037,0.125
Fare,0.018,0.269,0.670,1.0,0.070,0.266,0.368,0.0,0.394,1.000,0.999,0.166,0.0,1.000,0.070,0.070,0.266


## 10. 🔬 Data Integrity Testers (Statistical Suite)
The final assignment required robust verification tools for stationarity and serial independence. These tools verify the dataset's structural integrity.

In [26]:
# 1. MANOVA: Testing constant joint structural mean across chronological chunks
wrangler.test_constant_mean(columns=['Age', 'Fare'], chunks=4)

# 2. Box's M: Testing homoscedastic covariance stability across chunks
wrangler.test_constant_covariance(columns=['Age', 'Fare'], chunks=3)

# 3. Multivariate Ljung-Box: Testing for row serial independence (Autocorrelation)
wrangler.test_row_independence(columns=['Age', 'Fare'], max_lag=5)


--- MANOVA Mean Homogeneity Test (g=4 chunks, m=2 features) ---
Wilks' Lambda (Λ): 0.97457
Chi-Square Statistic: 18.2922 | Degrees of Freedom: 6
P-Value: 0.005542
🚨 Warning: Reject H0. Significant mean drift detected.



--- Box's M Covariance Homogeneity Test (g=3 chunks, m=2 features) ---
Box's M Statistic: 18.0677
Asymptotic Chi-Square: 17.9943 | Degrees of Freedom: 6
P-Value: 0.006247
✅ Success: Fail to reject H0. Covariance structure is stable.



--- Multivariate Ljung-Box Serial Independence Test (Lags Checked = 5) ---
Portmanteau Statistic Q_m(H): 17.3320
Degrees of Freedom: 20
P-Value: 0.631324
✅ Success: Fail to reject H0. Rows are independent.


{'Q_m': np.float64(17.331979205714248),
 'p_value': np.float64(0.6313235291943018),
 'df': 20}

## 11. 🎨 Standalone Modular Plotting
If you need manual, isolated plots (or HTML string components for a web dashboard), you can utilize the `PlottingMethods` wrapper.

In [27]:
from data_wrangling import PlottingMethods

# Generate a distplot for Age
fig = PlottingMethods.distplot(wrangler.df, col_names=['Age'], title='Titanic Age Distribution')
fig.show()

# Output can also be rendered as raw HTML!
html_str = PlottingMethods.histogram(wrangler.df, x_col='Age', color_col='Survived', as_html=True)
print(f"HTML String begins with: {html_str[:50]}...")

HTML String begins with: <div>                        <script type="text/ja...


/home/jazz/.local/lib/python3.14/site-packages/plotly/express/_core.py:1985: FutureWarning:

When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.



## 12. 💾 Data Extraction, Reset, & Export
Finally, you can slice out subsets of your wrangled data, save it back to disk, and securely wipe the engine's state.

In [28]:
# Extract numerical data safely into selected_df
subset = wrangler.extract_data(column_type='numerical')

# Export the cleaned state to a new CSV
wrangler.export_data(dataset='working', file_name='wrangled_titanic', export_summary=True)

# Securely reset engine for a new project
wrangler.reset_df()

✅ Data extracted into selected_df: 891 rows, 9 columns.


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Ticket,Fare,count
0,1,0,3,22.0,1,0,NaN,7.2500,1
1,2,1,1,38.0,1,0,NaN,71.2833,1
2,3,1,3,26.0,0,0,NaN,7.9250,1
3,4,1,1,35.0,1,0,113803.0,53.1000,1
4,5,0,3,35.0,0,0,373450.0,8.0500,1


💾 Data exported to wrangled_titanic_1780124653.csv
📄 Summary exported to wrangled_titanic_1780124653_summary.json
✅ DataFrame reset to original state.
--- Data Summary  ---


,Column Name,Data Type,Total Records,Missing Values
0,PassengerId,int64,891,0
1,Survived,int64,891,0
2,Pclass,int64,891,0
3,Name,object,891,0
4,Sex,object,891,0
5,Age,float64,891,177
6,SibSp,int64,891,0
7,Parch,int64,891,0
8,Ticket,float64,891,230
9,Fare,float64,891,0
